# 18b — The necessity test (E19): leave-EFG-out anchors (T2) and the conditional no-EFG ensemble (T3)

**VERSION = "v4"** (cell 1) runs on the re-registered design (`runs_v4/`, `spec/manifest_v4.csv`: seven scenarios × two refugia realizations = 14 formulations, after 12 and 18). What changed against v3.1 — every change keyed on a NEW manifest column, so `VERSION = "v3.1"` still reproduces the prior registration from its own manifest: the block structure for the leave-one-block-out anchors (E17 T3, cell 3) and the T3 guarded floors comes from the manifest's `block_structure` (five blocks under v4 → six arms: one per block + the EFG-out; the four PROACT blocks of `spec/e_round_v13.json` before), every block feature asserted present in the ingested stack; the refugia layer per formulation from `macrorefugia_path` (the SSP245 patch is no longer a literal), the ingested ssp585 layer asserted against it; the EFG block folder from `efg_block_version` (v3 → `iucn_efg_v3`); the manifest may carry 12 or 14 formulations. Run order for the v4 record: **11c → 12 → 13 → 15 → 18 → 18b → 18c → 19 → 20 → 21**, each run by Ethan in VS Code (no headless runs). Naming convention: things by what they are, codes in parentheses.

Runs on the curated-block re-solves only (`VERSION` ≠ `"v1"`). **T2** (~one solve × ≤ 20 min per formulation): for every design
formulation, the anchor solved with every EFG multiplier at 0, certified to a 1e-3 gap (the EFG-free objective is a near-flat
plateau; 1e-4 took 25 min in v1 and > 100 min on v3.1 — T2 is a witness test, so 1e-3 suffices; M4.28 addendum) (exactly as the leave-block-out anchors (E17 T3) did
for `efg_out`) → `runs<_version>/e19_t2/<formulation_id>/run/portfolio.tif`. Core cells absent from every no-EFG anchor are
EFG-necessary by counterfactual (18c compares them with the adequacy-forced set (E19 T1)).
**T3** (~9 h + a guarded sweep): the full no-EFG ensemble (anchor + 50 MGA members + 50 guarded members per formulation)
— runs ONLY if `spec/<version>/e19_gate.json`, written by 18c, says the core is predominantly forced (> 50%). Resumable;
live internet (WLS). Kernel `R (y2y)`.

In [ ]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # the ACTIVE version's stack manifest (aligned_stack_v4/manifest.json under v4)
VERSION <- "v4"      # v4 = manifest v4, study plan v0.20 -- five blocks, seven scenarios, floored refugia + squared transboundary current; v3.1 = the prior registration
stopifnot("the necessity test (E19) runs on the curated-block re-solves only (study plan v0.17.2)" = VERSION != "v1")
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION)
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
# the EFG block folder: the manifest's efg_block_version when it carries one (v3 -> iucn_efg_v3; shared by v3, v3.1 and v4),
# else the pre-column rule (v1 -> iucn_efg; minor versions share the block)
EFG_SUBDIR_EXPECTED <- if ("efg_block_version" %in% names(MAN)) {
  stopifnot("formulations disagree on efg_block_version -- STOP" = length(unique(MAN$efg_block_version)) == 1)
  paste0("iucn_efg_", MAN$efg_block_version[1])
} else if (VERSION == "v1") "iucn_efg" else paste0("iucn_efg_", sub("\\..*$", "", VERSION))
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) %in% c(12, 14))
RUNS <- file.path(PROJ, RUNS_REL)
# ---- the refugia layer per formulation -----------------------------------------------------------------------
# manifest v4 registers it per row (column macrorefugia_path: the floored 1/v layer of the v4 stack for ssp585, its 245
# realization for ssp245); earlier manifests carry no column, so the pre-v4 literals stand in.
REFUGIA_585_FALLBACK <- "input_data/aligned_stack/climate_type_macrorefugia.tif"
REFUGIA_245_FALLBACK <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
refugia_path_for <- function(row) {
  p <- if ("macrorefugia_path" %in% names(row)) trimws(as.character(row$macrorefugia_path)) else NA_character_
  if (length(p) == 1 && !is.na(p) && nzchar(p)) return(p)
  if (grepl("^ssp245", row$climate_level)) REFUGIA_245_FALLBACK else REFUGIA_585_FALLBACK
}
i245 <- which(grepl("^ssp245", MAN$climate_level))
paths245 <- unique(vapply(i245, function(i) refugia_path_for(MAN[i, ]), character(1)))
stopifnot("ssp245 formulations disagree on their refugia realization -- STOP" = length(paths245) <= 1)
REFUGIA_245_PATH <- if (length(paths245) == 1) paths245 else REFUGIA_245_FALLBACK   # patched into the 245 base context
cat(sprintf("refugia (ssp245, %d formulations): %s\n", length(i245), REFUGIA_245_PATH))
# ---- the value blocks --------------------------------------------------------------------------------------------
# the manifest's block_structure when it carries one (v4: five blocks -- core habitat, structural connectivity, climate
# corridors, carbon, biodiversity; asserted identical on every row), else the four PROACT blocks of the E-round record
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))   # E12 seeds (+ the pre-v4 blocks)
BLOCKS <- if ("block_structure" %in% names(MAN)) {
  stopifnot("formulations disagree on block_structure -- the manifest is not one design; STOP" = length(unique(MAN$block_structure)) == 1)
  lapply(jsonlite::fromJSON(MAN$block_structure[1]), unlist)
} else lapply(ER$e17_t3$blocks, unlist)
FLOOR_G <- 0.05
ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585)); ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))
ctx245 <- pr_setup(mpath, PROJ); ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REFUGIA_245_PATH
ctx245 <- modifyList(ctx245, pr_ingest(ctx245)); ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_names <- ctx585$layers$name[ctx585$layers$role == "feature_efg"]
stopifnot(length(efg_names) > 0, all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx585$layers$path[ctx585$layers$role == "feature_efg"])))
# the 585 context ingests whatever stack config.Y2Y_VERSION points at (aligned_stack_v4/ under v4): its refugia layer must
# be the one the manifest registers for the ssp585 formulations, else VERSION and config disagree
refugia_ingested <- ctx585$layers$path[ctx585$layers$name == "climate_type_macrorefugia"]
paths585 <- unique(vapply(which(!grepl("^ssp245", MAN$climate_level)), function(i) refugia_path_for(MAN[i, ]), character(1)))
same_file <- function(a, b) {
  abs_of <- function(p) normalizePath(if (grepl("^/", p)) p else file.path(PROJ, p), mustWork = FALSE)
  identical(abs_of(a), abs_of(b))
}
stopifnot("the ingested refugia layer is not the one the manifest registers for ssp585 -- config.Y2Y_VERSION and VERSION disagree; STOP" =
            length(paths585) == 1 && length(refugia_ingested) == 1 && same_file(paths585, refugia_ingested))
# every block feature must be a continuous feature the ingested stack carries (replaces the fixed four-name check)
cont_names <- ctx585$layers$name[ctx585$layers$role == "feature_continuous"]
stopifnot("a block names a feature the ingested stack does not carry -- block_structure vs manifest.json; STOP" =
            length(BLOCKS) > 0 && all(unlist(BLOCKS) %in% cont_names))
cat(sprintf("blocks (%d): %s | %d features verified against the ingested stack\n", length(BLOCKS),
            paste(names(BLOCKS), collapse = ", "), length(unlist(BLOCKS))))
base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))
no_efg <- function(w) { for (f in efg_names) w[[f]] <- 0; w }        # every EFG multiplier -> 0 (E17 T3 convention)
# T2 solver settings (M4.28 addendum, 2026-09-14): the leave-EFG-out objective is a near-flat plateau (D ~ 1), and certifying
# 1e-4 on it took 25 min in v1 and > 100 min on v3.1's first formulation. T2 is a WITNESS test (a no-EFG plan that keeps a
# cell proves it is not EFG-necessary), so anchors are certified to 1e-3 -- fifty times inside the 5% band -- with a
# 20-minute cap; the achieved gap and bound are recorded per anchor and read back by 18c. T1 and the T3 gate are unaffected.
T2_GAP <- 1e-3; T2_TIME_LIMIT_S <- 1200
run_single <- function(base_ctx, w, t, out_rel, artifact = "run", opt_gap = T2_GAP, time_limit = T2_TIME_LIMIT_S) {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_ctx, targets = t, feature_weight_multipliers = w,
      results_dir = out_rel, results_subdir = artifact, solver = "gurobi", decision_type = "binary", opt_gap = opt_gap,
      solver_time_limit = time_limit, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx)); pr_write_outputs(actx); invisible(NULL)
}
cat(sprintf("VERSION %s | %d design formulations | %d blocks (%s) | %d EFG features zeroed for the counterfactuals\n",
            VERSION, nrow(MAN), length(BLOCKS), paste(names(BLOCKS), collapse = ", "), length(efg_names)))


In [2]:
# ---- T2: leave-EFG-out anchors, one per design formulation (gap 1e-3, <= 20 min each) --------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; wt <- form_wt(row)
  cat(sprintf("== %s (%d/%d)\n", row$formulation_id, i, nrow(MAN)))
  run_single(base_for(row), no_efg(wt$w), wt$t, file.path(RUNS_REL, "e19_t2", row$formulation_id))
  rs <- file.path(RUNS, "e19_t2", row$formulation_id, "run", "run_summary.json")
  if (file.exists(rs)) {
    pv <- jsonlite::read_json(rs)$solver_provenance
    cat(sprintf("   status %s | objective %.4f | bound %.4f | gap %.1e | %.0f s\n", pv$status, as.numeric(pv$objective), as.numeric(pv$objbound), as.numeric(pv$gap), as.numeric(pv$runtime)))
  }
}
cat("T2 complete -- next: 18c_e19_analysis (T1 + T2 agreement + the T3 gate)\n")


== s0_ssp585_theta5 (1/12)
   analyses/y2y/runs_v3.1/e19_t2/s0_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 4.8995 | bound 4.8990 | gap 1.0e-04 | 41 s
== s1_ssp585_theta5 (2/12)
   analyses/y2y/runs_v3.1/e19_t2/s1_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 4.6994 | bound 4.6985 | gap 2.0e-04 | 40 s
== s2_ssp585_theta5 (3/12)
   analyses/y2y/runs_v3.1/e19_t2/s2_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 5.0615 | bound 5.0615 | gap 2.4e-08 | 47 s
== s3_ssp585_theta5 (4/12)
   analyses/y2y/runs_v3.1/e19_t2/s3_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 5.0904 | bound 5.0904 | gap 3.0e-07 | 45 s
== s4_ssp585_theta3 (5/12)
   analyses/y2y/runs_v3.1/e19_t2/s4_ssp585_theta3 exists -- skipped
   status OPTIMAL | objective 4.5173 | bound 4.5172 | gap 4.6e-06 | 49 s
== s5_ssp585_theta5 (6/12)
   analyses/y2y/runs_v3.1/e19_t2/s5_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 11.1269 | bound 11.1264 | gap 0.0e+

In [ ]:
# ---- E17 T3 on the manifest's blocks: leave-one-theme-out anchors at S0 (one per block + the EFG-out: six arms under v4, five under v3.1) --
# The E17 one-pager's bars were v1 evidence (runs/e17_t3, the 40-class block); each registration re-solves them on its own block
# structure: every block-out in names(BLOCKS) (v4: core habitat, structural connectivity, climate corridors, carbon, biodiversity)
# at the standard 1e-4 gap (15-67 s each in v1) and the EFG-out at the T2 witness gap.
row0 <- MAN[MAN$formulation_id == "s0_ssp585_theta5", ]; stopifnot(nrow(row0) == 1); wt0 <- form_wt(row0)
for (b in names(BLOCKS)) {
  w <- wt0$w; for (f in unlist(BLOCKS[[b]])) w[[f]] <- 0
  cat(sprintf("== %s OUT\n", b))
  run_single(ctx585, w, wt0$t, file.path(RUNS_REL, "e17_t3", paste0(b, "_out")), opt_gap = 1e-4, time_limit = 43200)
}
cat("== efg OUT\n")
run_single(ctx585, no_efg(wt0$w), wt0$t, file.path(RUNS_REL, "e17_t3", "efg_out"))
cat(sprintf("E17 T3 (%d block-outs + efg_out on the %s block structure) complete -- re-run 19 then 20 for the one-pager\n", length(BLOCKS), VERSION))

In [4]:
# ---- T3 (CONDITIONAL): the no-EFG ensemble -- anchors + MGA + guarded members, EFG multipliers 0 --------------
gate_f <- file.path(PROJ, sprintf("analyses/y2y/spec/%s/e19_gate.json", VERSION))
gate <- if (file.exists(gate_f)) jsonlite::read_json(gate_f) else NULL
if (is.null(gate)) {
  cat("T3 gate not written yet -- run 18c_e19_analysis first (it decides whether the core is predominantly forced)\n")
} else if (!isTRUE(gate$t3_triggered)) {
  cat(sprintf("T3 NOT triggered: forced share of the core %.1f%% (rule: > 50%%) -- the no-EFG ensemble is not run\n", 100 * gate$forced_share_core_all))
} else {
  cat(sprintf("T3 TRIGGERED: forced share of the core %.1f%% -- solving the no-EFG ensemble (~9 h + guarded)\n", 100 * gate$forced_share_core_all))
  for (i in seq_len(nrow(MAN))) {
    row <- MAN[i, ]; wt <- form_wt(row); cd <- file.path(RUNS, "e19_t3", row$formulation_id); dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
    if (file.exists(file.path(cd, "mga_guard_g05.tif"))) { cat("   exists -- skipped\n"); next }
    actx <- pr_override(base_for(row), targets = wt$t, feature_weight_multipliers = no_efg(wt$w),
        results_dir = file.path(RUNS_REL, "e19_t3", row$formulation_id), results_subdir = "mga_build",
        solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
    actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
    bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
    cm <- mga_compile(actx); anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
    if (!file.exists(file.path(cd, "mga_g05.tif"))) {
      gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g05")
    }
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(formulation_id = row$formulation_id, experiment = "E19 T3 no-EFG ensemble", anchor_objective = anchor$z,
                              anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, weight_vector = no_efg(wt$w), target_vector = wt$t,
                              k = row$k_requested, g = row$band_gap_g, created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    gg <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    layers <- lapply(seq_len(gg$k), function(j) { rr <- terra::rast(actx$cost); vv <- rep(NA_integer_, terra::ncell(rr)); vv[cm$pu_index] <- as.integer(gg$members[j, ]); terra::values(rr) <- vv; rr })
    s <- terra::rast(layers); names(s) <- sprintf("guard_%02d", seq_len(gg$k))
    terra::writeRaster(s, file.path(cd, "mga_guard_g05.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    write.csv(gg$certificates, file.path(cd, "certificates_guard.csv"), row.names = FALSE)
    cat(sprintf("   wrote anchor, MGA members, guarded members for %s\n", row$formulation_id))
  }
  cat("T3 complete -- re-run 18c_e19_analysis for the F_noEFG surface\n")
}


T3 NOT triggered: forced share of the core 0.0% (rule: > 50%) -- the no-EFG ensemble is not run
